# Limpieza de datos: Conteos Censales 2009-2024

Lee `Datos/datos_procesados/concatenacion_2009-2024.csv` y deja una tabla limpia y agregada: **una fila por año × estado × municipio × sexo × rango de edad × tipo de sección**.

Decisiones aplicadas:
- Se eliminan los registros con `SEXO = 2` y con `EDAD > 122`.
- `EDAD` pasa a rangos: 18-29, 30-39, 40-49, 50-59, 60-69, 70-79, 80-89 y 90 o más.
- `FELECCION` se guarda como texto `dd/mm/yyyy` (cada año traía su propio formato).
- Se agrega `TIPO_ELECCION` (Presidencial / Intermedia).
- `SEXO` pasa a Hombre / Mujer y `TIPOSEC` a Urbano / Rural / Mixto.
- `SECCION`, `DEF` y `LN` no se leen: la unidad es el municipio y `LN = SV + NV + NS`.
- Nombres de estados y municipios normalizados, más un diccionario de 31 municipios.
- Al final: `ABSTENCION = NV / (SV + NV)` y `CALIDAD_NS = NS / (SV + NV + NS)`.

**Salidas:** `Datos/datos_procesados/base_limpia.csv` y `Datos/datos_procesados/diccionario_municipios.csv`.

## 1. Configuración y rutas

In [127]:
from pathlib import Path
import pandas as pd

_cwd = Path.cwd().resolve()
RAIZ = next(p for p in [_cwd, *_cwd.parents] if (p / "Datos").is_dir())
SALIDA = RAIZ / "Datos" / "datos_procesados"
ARCHIVO = SALIDA / "concatenacion_2009-2024.csv"
RUTA_BASE = SALIDA / "base_limpia.csv"
RUTA_DICCIONARIO = SALIDA / "diccionario_municipios.csv"

print("Archivo:", ARCHIVO, "| existe:", ARCHIVO.exists())

Archivo: C:\Users\camil\Documents\Programming\INE\Datos\datos_procesados\concatenacion_2009-2024.csv | existe: True


## 2. Lectura del CSV
Solo se leen las columnas necesarias (sin `SECCION`, `DEF` ni `LN`). Los textos con pocos valores distintos se leen como `category` y los números como enteros pequeños para que el archivo quepa en memoria.

In [128]:
TIPOS = {
    "AELEC": "int16", "FELECCION": "category", "EDONOM": "category", "MPIONOM": "category",
    "TIPOSEC": "category", "SEXO": "int8", "EDAD": "int16",
    "SV": "int16", "NV": "int16", "NS": "int16",
}

data = pd.read_csv(ARCHIVO, usecols=list(TIPOS), dtype=TIPOS)
print(f"{len(data):,} filas | memoria: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB")
data.head()

54,872,712 filas | memoria: 0.88 GB


,AELEC,FELECCION,EDONOM,MPIONOM,TIPOSEC,SEXO,EDAD,SV,NV,NS
0,2009,5/7/2009,AGUASCALIENTES,Aguascalientes,U,0,18,11,20,0
1,2009,5/7/2009,AGUASCALIENTES,Aguascalientes,U,0,19,7,13,0
2,2009,5/7/2009,AGUASCALIENTES,Aguascalientes,U,0,20,9,19,0
3,2009,5/7/2009,AGUASCALIENTES,Aguascalientes,U,0,21,9,12,0
4,2009,5/7/2009,AGUASCALIENTES,Aguascalientes,U,0,22,12,18,0


## 3. Totales de partida
Se guardan las filas y las personas (`SV + NV + NS`) iniciales para comprobar al final que los totales cuadran. `registro` anota lo que se elimina en cada paso.

In [129]:
FILAS_INICIAL = len(data)
PERSONAS_INICIAL = data[["SV", "NV", "NS"]].sum().sum()

registro = []   # lo que se elimina en cada paso

print(f"Filas: {FILAS_INICIAL:,} | Personas (SV + NV + NS): {PERSONAS_INICIAL:,}")

Filas: 54,872,712 | Personas (SV + NV + NS): 518,522,733


## 4. Eliminar registros con sexo sin definir (`SEXO = 2`)

In [130]:
fuera = data["SEXO"] == 2
registro.append({"paso": "SEXO = 2", "filas": fuera.sum(), "personas": data.loc[fuera, ["SV", "NV", "NS"]].sum().sum()})
data = data[~fuera]

print(f"Eliminadas: {fuera.sum():,} filas | restantes: {len(data):,}")

Eliminadas: 105 filas | restantes: 54,872,607


## 5. Eliminar edades imposibles (`EDAD > 122`)
Son filas de una sola persona con edades superiores al récord de longevidad. Las edades de 91 a 122 se conservan y se agrupan en "90 o más" en el siguiente paso.

In [131]:
fuera = data["EDAD"] > 122
registro.append({"paso": "EDAD > 122", "filas": fuera.sum(), "personas": data.loc[fuera, ["SV", "NV", "NS"]].sum().sum()})
data = data[~fuera]

print(f"Eliminadas: {fuera.sum():,} filas | restantes: {len(data):,}")

Eliminadas: 286 filas | restantes: 54,872,321


## 6. Edad en rangos
No hay edades menores de 18. Cada rango incluye el número final (por ejemplo, 18-29 llega hasta los 29 años).

In [132]:
data["RANGO_EDAD"] = pd.cut(
    data["EDAD"],
    bins=[17, 29, 39, 49, 59, 69, 79, 89, 122],
    labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80-89", "90 o más"],
)
data = data.drop(columns="EDAD")

print(data["RANGO_EDAD"].value_counts().sort_index())

RANGO_EDAD
18-29       9566476
30-39       7988474
40-49       7951802
50-59       7858485
60-69       7603956
70-79       6960688
80-89       5110187
90 o más    1832253
Name: count, dtype: int64


## 7. Sexo: de código a texto
En esta base `0 = Hombre` y `1 = Mujer` (el código 1 concentra 51.9 % de la lista nominal, en línea con la lista nacional). La segunda salida lo comprueba: las mujeres deben participar más que los hombres.

In [133]:
data["SEXO"] = data["SEXO"].map({0: "Hombre", 1: "Mujer"}).astype("category")

t = data.groupby("SEXO", observed=True)[["SV", "NV"]].sum()
print("Participación SV / (SV + NV) por sexo:")
print((t["SV"] / (t["SV"] + t["NV"])).round(3))

Participación SV / (SV + NV) por sexo:
SEXO
Hombre    0.505
Mujer     0.586
dtype: float64


## 8. Fecha de la elección y tipo de elección
Cada año traía su formato: 2009 a 2021 venían día/mes y 2024 venía mes/día (`06/02/2024` es el 2 de junio). Por eso se traduce cada texto a mano a `dd/mm/yyyy`. Antes de borrar `AELEC`, se usa para crear `TIPO_ELECCION`.

In [134]:
print("Textos originales:", list(data["FELECCION"].cat.categories))

FECHAS = {
    "5/7/2009": "05/07/2009",
    "1/7/2012": "01/07/2012",
    "7/6/2015": "07/06/2015",
    "1/7/2018": "01/07/2018",
    "6/6/2021": "06/06/2021",
    "06/02/2024": "02/06/2024",
}
data["FELECCION"] = data["FELECCION"].map(FECHAS)

print("Fechas sin traducir (debe ser 0):", data["FELECCION"].isna().sum())

Textos originales: ['5/7/2009', '1/7/2012', '7/6/2015', '1/7/2018', '6/6/2021', '06/02/2024']
Fechas sin traducir (debe ser 0): 0


In [135]:
TIPO = {
    2009: "Intermedia", 2012: "Presidencial", 2015: "Intermedia",
    2018: "Presidencial", 2021: "Intermedia", 2024: "Presidencial",
}
data["TIPO_ELECCION"] = data["AELEC"].map(TIPO).astype("category")

# Cada año debe tener una sola fecha y un solo tipo
print(data.groupby(["AELEC", "FELECCION", "TIPO_ELECCION"], observed=True).size())

AELEC  FELECCION   TIPO_ELECCION
2009   05/07/2009  Intermedia       8706684
2012   01/07/2012  Presidencial     8604415
2015   07/06/2015  Intermedia       9106059
2018   01/07/2018  Presidencial     9251079
2021   06/06/2021  Intermedia       9430473
2024   02/06/2024  Presidencial     9773611
dtype: int64


In [136]:
data.head()

,AELEC,FELECCION,EDONOM,MPIONOM,TIPOSEC,SEXO,SV,NV,NS,RANGO_EDAD,TIPO_ELECCION
0,2009,05/07/2009,AGUASCALIENTES,Aguascalientes,U,Hombre,11,20,0,18-29,Intermedia
1,2009,05/07/2009,AGUASCALIENTES,Aguascalientes,U,Hombre,7,13,0,18-29,Intermedia
2,2009,05/07/2009,AGUASCALIENTES,Aguascalientes,U,Hombre,9,19,0,18-29,Intermedia
3,2009,05/07/2009,AGUASCALIENTES,Aguascalientes,U,Hombre,9,12,0,18-29,Intermedia
4,2009,05/07/2009,AGUASCALIENTES,Aguascalientes,U,Hombre,12,18,0,18-29,Intermedia


## 9. Tipo de sección: de código a texto

In [137]:
data["TIPOSEC"] = data["TIPOSEC"].map({"U": "Urbano", "R": "Rural", "M": "Mixto"})

print("Sin traducir (debe ser 0):", data["TIPOSEC"].isna().sum())
print(data["TIPOSEC"].value_counts())

Sin traducir (debe ser 0): 0
TIPOSEC
Urbano    35056100
Rural     14584106
Mixto      5232115
Name: count, dtype: int64


## 10. Nombres de estados y municipios: normalización
Mayúsculas, sin acentos, sin puntuación y sin espacios de más. Se aplica a los nombres distintos (pocos), no a los 54 millones de filas. `DISTRITO FEDERAL` pasa a `CIUDAD DE MEXICO`.

In [138]:
import re
import unicodedata

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()   # quita acentos
    texto = re.sub(r"[^A-Z0-9 ]", "", texto.upper())                                  # solo letras, números y espacios
    return " ".join(texto.split())                                                    # quita espacios de más

ESTADOS = {c: normalizar(c) for c in data["EDONOM"].cat.categories}
ESTADOS["DISTRITO FEDERAL"] = "CIUDAD DE MEXICO"
data["EDONOM"] = data["EDONOM"].map(ESTADOS).astype("category")

MUNICIPIOS = {c: normalizar(c) for c in data["MPIONOM"].cat.categories}
data["MPIONOM"] = data["MPIONOM"].map(MUNICIPIOS).astype("category")

print("Estados:", data["EDONOM"].nunique())
print("Nombres de municipio distintos:", data["MPIONOM"].nunique())

Estados: 32
Nombres de municipio distintos: 2364


## 11. Agregación
Se suman `SV`, `NV` y `NS` por elección × tipo de elección × estado × municipio × sexo × rango de edad × tipo de sección. Ya no hay `SECCION`: cada fila es un grupo dentro de un municipio. Las tasas se calculan **después** de sumar. Se libera `data` para ahorrar memoria.

In [139]:
CLAVE = ["AELEC", "FELECCION", "TIPO_ELECCION", "ESTADO", "MUNICIPIO", "SEXO", "RANGO_EDAD", "TIPOSEC"]

data = data.rename(columns={"EDONOM": "ESTADO", "MPIONOM": "MUNICIPIO"})
base = data.groupby(CLAVE, observed=True, as_index=False)[["SV", "NV", "NS"]].sum()
del data

print(f"De {FILAS_INICIAL:,} filas a {len(base):,}")
base.head()

De 54,872,712 filas a 544,833


,AELEC,FELECCION,TIPO_ELECCION,ESTADO,MUNICIPIO,SEXO,RANGO_EDAD,TIPOSEC,SV,NV,NS
0,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Mixto,765,2544,3512
1,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Rural,1002,2982,0
2,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Urbano,17777,45137,9191
3,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Mixto,723,1580,2814
4,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Rural,1067,2074,0


## 12. Diccionario de municipios
Une los nombres distintos de un mismo municipio (typos, abreviaturas y cambios de nombre entre años). Cada cambio lleva su estado, porque un mismo nombre puede existir en dos estados (por ejemplo `CARMEN`). Se aplica sobre la tabla ya agregada, que es pequeña, y después se vuelve a sumar para juntar las filas que ahora coinciden. Los dos pares de Mixtepec de Oaxaca (San Juan y San Pedro) quedan combinados.

In [140]:
CAMBIOS = [
    # (estado, nombre en la base, nombre final)
    ("CAMPECHE", "DZITBLACHE", "DZITBALCHE"),
    ("CAMPECHE", "HOPELCHEEN", "HOPELCHEN"),
    ("CHIAPAS", "HUITIUPAAN", "HUITIUPAN"),
    ("CHIAPAS", "MARAVILLA TEJENAPA", "MARAVILLA TENEJAPA"),
    ("CHIAPAS", "VILLACOMALTITLAN", "VILLA COMALTITLAN"),
    ("CHIHUAHUA", "BATOPILAS", "BATOPILAS DE MANUEL GOMEZ MORIN"),
    ("COAHUILA", "CUATROCIENEGAS", "CUATRO CIENEGAS"),
    ("DURANGO", "GENERAL SIMON BOLIVAR", "SIMON BOLIVAR"),
    ("HIDALGO", "SAN AGUSTON METZQUITITLAN", "SAN AGUSTIN METZQUITITLAN"),
    ("MORELOS", "JONACATEPEC DE LEANDRO VALLE", "JONACATEPEC"),
    ("NUEVO LEON", "DR ARROYO", "DOCTOR ARROYO"),
    ("NUEVO LEON", "DR COSS", "DOCTOR COSS"),
    ("NUEVO LEON", "DR GONZALEZ", "DOCTOR GONZALEZ"),
    ("NUEVO LEON", "CARMEN", "EL CARMEN"),
    ("NUEVO LEON", "GRAL BRAVO", "GENERAL BRAVO"),
    ("NUEVO LEON", "GRAL ESCOBEDO", "GENERAL ESCOBEDO"),
    ("NUEVO LEON", "GRAL TERAN", "GENERAL TERAN"),
    ("NUEVO LEON", "GRAL TREVINO", "GENERAL TREVINO"),
    ("NUEVO LEON", "GRAL ZARAGOZA", "GENERAL ZARAGOZA"),
    ("NUEVO LEON", "GRAL ZUAZUA", "GENERAL ZUAZUA"),
    ("OAXACA", "JUCHITAN DE ZARAGOZA", "HEROICA CIUDAD DE JUCHITAN DE ZARAGOZA"),
    ("OAXACA", "SAN BLAS ATEMPA", "HEROICA VILLA DE SAN BLAS ATEMPA"),
    ("OAXACA", "HEROICA VILLA TEZOATLAN DE SEGURA Y LUNA CUNA DE",
               "HEROICA VILLA TEZOATLAN DE SEGURA Y LUNA CUNA DE LA INDEPENDENCIA DE OAXACA"),
    ("OAXACA", "SAN JUAN MIXTEPEC DTO 08", "SAN JUAN MIXTEPEC"),
    ("OAXACA", "SAN JUAN MIXTEPEC DTO 26", "SAN JUAN MIXTEPEC"),
    ("OAXACA", "SAN PEDRO MIXTEPEC DTO 22", "SAN PEDRO MIXTEPEC"),
    ("OAXACA", "SAN PEDRO MIXTEPEC DTO 26", "SAN PEDRO MIXTEPEC"),
    ("OAXACA", "VILLA DE SANTIAGO CHAZUMBA", "SANTIAGO CHAZUMBA"),
    ("OAXACA", "VILLA DE TUTUTEPEC DE MELCHOR OCAMPO", "VILLA DE TUTUTEPEC"),
    ("OAXACA", "VILLA HIDALGO", "VILLA HIDALGO YALALAG"),
    ("TLAXCALA", "ZILTLALTEPEC DE TRINIDAD SANCHEZ SANTOS", "ZITLALTEPEC DE TRINIDAD SANCHEZ SANTOS"),
    ("VERACRUZ", "ALTO LUCERO DE GUTIERREZ BARIOS", "ALTO LUCERO DE GUTIERREZ BARRIOS"),
    ("VERACRUZ", "COSAMALOAPAN DE CARPIO", "COSAMALOAPAN"),
]

base["ESTADO"] = base["ESTADO"].astype(str)
base["MUNICIPIO"] = base["MUNICIPIO"].astype(str)

for estado, actual, final in CAMBIOS:
    filas = (base["ESTADO"] == estado) & (base["MUNICIPIO"] == actual)
    if not filas.any():
        print("No se encontró:", estado, "-", actual)
    base.loc[filas, "MUNICIPIO"] = final

# Las filas que ahora tienen el mismo nombre se suman en una sola
base = base.groupby(CLAVE, observed=True, as_index=False)[["SV", "NV", "NS"]].sum()

pd.DataFrame(CAMBIOS, columns=["ESTADO", "NOMBRE_ORIGINAL", "NOMBRE_FINAL"]).to_csv(
    RUTA_DICCIONARIO, index=False, encoding="utf-8-sig")

municipios = base[["ESTADO", "MUNICIPIO"]].drop_duplicates().shape[0]
print(f"{len(CAMBIOS)} nombres unificados | municipios: {municipios:,} (esperado: 2,475)")

33 nombres unificados | municipios: 2,475 (esperado: 2,475)


## 13. Abstención y calidad del dato
- `ABSTENCION = NV / (SV + NV)`: `NS` queda fuera porque no sabemos si esas personas votaron.
- `CALIDAD_NS = NS / (SV + NV + NS)`: qué proporción del grupo no tiene información.

In [141]:
base["ABSTENCION"] = (base["NV"] / (base["SV"] + base["NV"])).round(4)
base["CALIDAD_NS"] = (base["NS"] / (base["SV"] + base["NV"] + base["NS"])).round(4)

base[["ABSTENCION", "CALIDAD_NS"]].describe()

,ABSTENCION,CALIDAD_NS
count,540024.000000,544801.000000
mean,0.430194,0.045936
std,0.214777,0.135777
min,0.000000,0.000000
25%,0.269800,0.000000
50%,0.403000,0.000000
75%,0.564900,0.009900
max,1.000000,1.000000


## 14. Validaciones

In [142]:
anio = base["FELECCION"].str[-4:]

personas_final = base[["SV", "NV", "NS"]].sum().sum()
personas_esperadas = PERSONAS_INICIAL - sum(r["personas"] for r in registro)
print(f"Personas: {personas_final:,} | esperadas: {personas_esperadas:,} | cuadra: {personas_final == personas_esperadas}")

print("\nQué se eliminó:")
display(pd.DataFrame(registro))

print("Estados:")
print(base.groupby(anio)["ESTADO"].nunique())

print("\nMunicipios por elección:")
print(base.assign(ANIO=anio)[["ANIO", "ESTADO", "MUNICIPIO"]].drop_duplicates().groupby("ANIO").size())

print("\nNulos por columna:")
print(base.isna().sum())

print("\nFilas repetidas en la clave (debe ser 0):", base.duplicated(CLAVE).sum())

print("\nMunicipios según en cuántas elecciones aparecen:")
print(base.assign(ANIO=anio)[["ANIO", "ESTADO", "MUNICIPIO"]].drop_duplicates()
          .groupby(["ESTADO", "MUNICIPIO"]).size().value_counts().sort_index())

Personas: 518,522,342 | esperadas: 518,522,342 | cuadra: True

Qué se eliminó:


,paso,filas,personas
0,SEXO = 2,105,105
1,EDAD > 122,286,286


Estados:
FELECCION
2009    32
2012    32
2015    32
2018    32
2021    32
2024    32
Name: ESTADO, dtype: int64

Municipios por elección:
ANIO
2009    2456
2012    2453
2015    2456
2018    2461
2021    2466
2024    2475
dtype: int64

Nulos por columna:
AELEC               0
FELECCION           0
TIPO_ELECCION       0
ESTADO              0
MUNICIPIO           0
SEXO                0
RANGO_EDAD          0
TIPOSEC             0
SV                  0
NV                  0
NS                  0
ABSTENCION       4777
CALIDAD_NS          0
dtype: int64

Filas repetidas en la clave (debe ser 0): 0

Municipios según en cuántas elecciones aparecen:
1       9
2       5
3       5
5       3
6    2453
Name: count, dtype: int64


## 15. Eliminar grupos sin dato de abstención
Son grupos donde todos los registros son `NS` (`SV + NV = 0`), así que no se puede calcular la abstención. Se anotan en `registro` para que las validaciones sigan cuadrando.


In [143]:
nulos = base["ABSTENCION"].isna()
print(f"Filas con ABSTENCION vacía: {nulos.sum():,} de {len(base):,} ({nulos.mean():.2%})")
print(f"Personas (NS) en esas filas: {base.loc[nulos, 'NS'].sum():,}")

base = base.dropna(subset=["ABSTENCION"])

Filas con ABSTENCION vacía: 4,777 de 544,801 (0.88%)
Personas (NS) en esas filas: 668,006


In [144]:
print(f"Filas: {len(base):,}")
print(f"Personas (SV + NV + NS): {base[['SV', 'NV', 'NS']].sum().sum():,}")
print(f"Personas con dato (SV + NV): {base[['SV', 'NV']].sum().sum():,}")

Filas: 540,024
Personas (SV + NV + NS): 517,854,336
Personas con dato (SV + NV): 494,910,117


## 16. Exportar la base limpia
Se guarda con `utf-8-sig` para que Excel lea bien los acentos (por ejemplo "90 o más").

In [145]:
COLUMNAS_FINALES = CLAVE + ["SV", "NV", "NS", "ABSTENCION", "CALIDAD_NS"]
base[COLUMNAS_FINALES].to_csv(RUTA_BASE, index=False, encoding="utf-8-sig")

print(f"Guardado: {RUTA_BASE} | {len(base):,} filas")
base[COLUMNAS_FINALES].head()

Guardado: C:\Users\camil\Documents\Programming\INE\Datos\datos_procesados\base_limpia.csv | 540,024 filas


,AELEC,FELECCION,TIPO_ELECCION,ESTADO,MUNICIPIO,SEXO,RANGO_EDAD,TIPOSEC,SV,NV,NS,ABSTENCION,CALIDAD_NS
0,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Mixto,765,2544,3512,0.7688,0.5149
1,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Rural,1002,2982,0,0.7485,0.0000
2,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Urbano,17777,45137,9191,0.7174,0.1275
3,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Mixto,723,1580,2814,0.6861,0.5499
4,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Rural,1067,2074,0,0.6603,0.0000
